In [1]:
# Network configuration
BROKER_HOST = "localhost"
BROKER_PORT = 1883
BASE_TOPIC = "iiot/site-1"
HIGH_TEMP_C = 35.0
SENSOR_IDS = ("sensor-01", "sensor-02")


In [2]:
import json
import time
from datetime import datetime, timezone
import paho.mqtt.client as mqtt

readings = {}
gateway_connected = False

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def on_connect(client, userdata, flags, reason_code, properties):
    global gateway_connected
    if reason_code == 0:
        gateway_connected = True
        client.subscribe(f"{BASE_TOPIC}/sensors/+/temperature", qos=1)
        print("Gateway connected; subscribed to sensor temperature topics.")
    else:
        print(f"Gateway connection failed: {reason_code}")

def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode("utf-8"))
        sensor_id = payload["sensor_id"]
        temperature = float(payload["temperature_c"])
        if sensor_id not in SENSOR_IDS or not (-80 <= temperature <= 200):
            raise ValueError("unknown sensor or implausible temperature")
    except (UnicodeDecodeError, json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
        print(f"Ignoring invalid message on {msg.topic}: {exc}")
        return

    readings[sensor_id] = temperature
    status_code = int(any(value >= HIGH_TEMP_C for value in readings.values()))
    status = {
        "gateway_id": "gateway-01",
        "status_code": status_code,
        "status": "HIGH_TEMPERATURE" if status_code else "NORMAL",
        "threshold_c": HIGH_TEMP_C,
        "sensor_readings_c": dict(readings),
        "updated_at": utc_now(),
    }
    client.publish(f"{BASE_TOPIC}/gateway/status", json.dumps(status), qos=1, retain=True)
    print(f"{sensor_id}: {temperature:.1f} °C → status_code={status_code} ({status['status']})")


In [4]:
def make_client(client_id):
    # Callback API v2 is supported by current Paho releases.
    return mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=client_id)

def publish_sensor_sequence(sensor_id, values_c, interval_s=1.0):
    client = make_client(sensor_id)
    client.connect(BROKER_HOST, BROKER_PORT, keepalive=30)
    client.loop_start()
    try:
        for temperature_c in values_c:
            message = {"sensor_id": sensor_id, "temperature_c": float(temperature_c), "timestamp": utc_now()}
            info = client.publish(
                f"{BASE_TOPIC}/sensors/{sensor_id}/temperature",
                json.dumps(message), qos=1
            )
            info.wait_for_publish(timeout=5)
            print(f"{sensor_id} published {temperature_c:.1f} °C")
            time.sleep(interval_s)
    finally:
        client.loop_stop()
        client.disconnect()

def run_demo():
    gateway = make_client("gateway-01")
    gateway.on_connect = on_connect
    gateway.on_message = on_message
    gateway.connect(BROKER_HOST, BROKER_PORT, keepalive=30)
    gateway.loop_start()
    try:
        deadline = time.time() + 10
        while not gateway_connected and time.time() < deadline:
            time.sleep(0.1)
        if not gateway_connected:
            raise TimeoutError("Gateway did not connect; check that the broker is running and reachable.")
        # Interleaved readings demonstrate normal and high states from both nodes.
        import threading
        nodes = [
            threading.Thread(target=publish_sensor_sequence, args=("sensor-01", [24.5, 36.2, 33.0], 0.7)),
            threading.Thread(target=publish_sensor_sequence, args=("sensor-02", [27.1, 29.4, 30.0], 0.9)),
        ]
        for node in nodes: node.start()
        for node in nodes: node.join()
        time.sleep(1)  # allow final gateway status publish to complete
    finally:
        gateway.loop_stop()
        gateway.disconnect()

# Run this cell to connect to the broker and send the example readings.
run_demo()


Gateway connected; subscribed to sensor temperature topics.
sensor-01 published 24.5 °C
sensor-01: 24.5 °C → status_code=0 (NORMAL)
sensor-02 published 27.1 °C
sensor-02: 27.1 °C → status_code=0 (NORMAL)
sensor-01 published 36.2 °C
sensor-01: 36.2 °C → status_code=1 (HIGH_TEMPERATURE)
sensor-02 published 29.4 °C
sensor-02: 29.4 °C → status_code=1 (HIGH_TEMPERATURE)
sensor-01 published 33.0 °C
sensor-01: 33.0 °C → status_code=0 (NORMAL)
sensor-02 published 30.0 °C
sensor-02: 30.0 °C → status_code=0 (NORMAL)


In [5]:
def status_code_for(sensor_values, threshold_c=HIGH_TEMP_C):
    return int(any(float(value) >= threshold_c for value in sensor_values.values()))

assert status_code_for({"sensor-01": 22.0, "sensor-02": 34.9}) == 0
assert status_code_for({"sensor-01": 35.0, "sensor-02": 28.0}) == 1
assert status_code_for({"sensor-01": 22.0, "sensor-02": 38.0}) == 1
print("Offline threshold logic checks passed.")


Offline threshold logic checks passed.
